# Resultados e Discussão — Sistema Híbrido RAG + Autoencoder

Síntese final do projeto: consolida os resultados quantitativos das duas partes (RAG documental — notebooks 01/02; Autoencoder de Gamma Ray — notebook 03) e apresenta a discussão crítica geral, conforme exigido pelo enunciado (baseline, métricas, análise qualitativa, discussão).

## 1. Resumo executivo

**Proposta**: sistema híbrido RAG + Autoencoder para consulta documental e reconstrução de curvas Gamma Ray em perfis convencionais de poços da ANP.

- **Parte 1 (RAG)**: 88 poços com perfil composto (PDF), metadados extraídos via OCR, 264 chunks textuais indexados. Avaliado com 23 perguntas (10 exatas, 8 semânticas, 5 mistas) contra 4 baselines: LLM sem RAG, TF-IDF, BM25, sistema híbrido (metadados estruturados + busca vetorial).
- **Parte 2 (Autoencoder)**: 73 poços com curva Gamma Ray utilizável, Autoencoder Conv1D treinado de forma unificada (59 poços treino / 14 teste, split por poço), comparado com PCA e média móvel.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import json
import pandas as pd

## 2. Resultados — Parte 1 (RAG)

Tabelas reproduzidas do notebook `02_rag_retrieval_evaluation.ipynb` (execute-o antes se quiser gerar os números do zero).

In [ ]:
from rag.metadata_loader import load_metadata_list
from rag.chunking import load_chunks
from rag.retrieval import TfidfRetriever, BM25Retriever, VectorRetriever, reciprocal_rank_fusion, structured_lookup
from rag.embeddings import embed_passages, embed_queries
from rag.qa_pipeline import detect_exact_field, extract_well_id, answer_structured
from evaluation.rag_metrics import evaluate_retrieval, evaluate_exact_answers

metadata_list = load_metadata_list(PROJECT_ROOT / "data" / "processed" / "metadata.json")
chunks = load_chunks(PROJECT_ROOT / "data" / "processed" / "chunks.jsonl")
questions = json.load(open(PROJECT_ROOT / "data" / "processed" / "questions_eval.json", encoding="utf-8"))
known_wells = [m["well_id"] for m in metadata_list]

tfidf_retriever = TfidfRetriever(chunks)
bm25_retriever = BM25Retriever(chunks)
chunk_embeddings = embed_passages([c["text"] for c in chunks])
vector_retriever = VectorRetriever(chunks, chunk_embeddings)

print(f"{len(metadata_list)} poços | {len(chunks)} chunks | {len(questions)} perguntas")

C:\Users\Visitante\Documents\well_rag_autoenconder\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

88 poços | 264 chunks | 23 perguntas


In [ ]:
### 2.1 Métricas de recuperação
retrieval_questions = [q for q in questions if q.get("well_id_esperado")]
query_embeddings = embed_queries([q["pergunta"] for q in retrieval_questions])

results_by_method = {"TF-IDF": [], "BM25": [], "Vetorial": [], "Híbrido (RRF)": []}
for i, q in enumerate(retrieval_questions):
    tfidf_r = tfidf_retriever.search(q["pergunta"], top_k=5)
    bm25_r = bm25_retriever.search(q["pergunta"], top_k=10)
    vector_r = vector_retriever.search(query_embeddings[i], top_k=5)
    hybrid_r = reciprocal_rank_fusion([bm25_r, vector_r], top_k=5)
    results_by_method["TF-IDF"].append([r.well_id for r in tfidf_r])
    results_by_method["BM25"].append([r.well_id for r in bm25_r[:5]])
    results_by_method["Vetorial"].append([r.well_id for r in vector_r])
    results_by_method["Híbrido (RRF)"].append([r.well_id for r in hybrid_r])

relevant = [{q["well_id_esperado"]} for q in retrieval_questions]
retrieval_rows = []
for method, retrieved in results_by_method.items():
    scores = evaluate_retrieval(retrieved, relevant, k_values=(1, 3, 5))
    retrieval_rows.append({
        "método": method, "Recall@1": scores.recall_at_k[1], "Recall@3": scores.recall_at_k[3],
        "Recall@5": scores.recall_at_k[5], "MRR": scores.mrr, "nDCG@5": scores.ndcg_at_k[5],
    })

retrieval_summary = pd.DataFrame(retrieval_rows)
print("Métricas de recuperação (13 perguntas com well_id conhecido):")
retrieval_summary

Métricas de recuperação (13 perguntas com well_id conhecido):


,método,Recall@1,Recall@3,Recall@5,MRR,nDCG@5
0,TF-IDF,1.0,1.0,1.0,1.0,1.0
1,BM25,1.0,1.0,1.0,1.0,1.0
2,Vetorial,1.0,1.0,1.0,1.0,1.0
3,Híbrido (RRF),1.0,1.0,1.0,1.0,1.0


In [ ]:
### 2.2 Métricas de resposta (perguntas de campo exato)
exact_questions = [q for q in questions if q["tipo"] == "exata"]

no_rag_answers = ["Não tenho informação suficiente para responder com precisão." for _ in exact_questions]
tfidf_answers = [(tfidf_retriever.search(q["pergunta"], top_k=1) or [None])[0] for q in exact_questions]
tfidf_answers = [r.text if r else "" for r in tfidf_answers]
bm25_answers = [(bm25_retriever.search(q["pergunta"], top_k=1) or [None])[0] for q in exact_questions]
bm25_answers = [r.text if r else "" for r in bm25_answers]

hybrid_answers = []
for q in exact_questions:
    well_id = q.get("well_id_esperado") or extract_well_id(q["pergunta"], known_wells)
    field = q.get("campo_esperado") or detect_exact_field(q["pergunta"])
    if well_id and field and "_e_" not in field:
        matches = structured_lookup(metadata_list, well_id=well_id)
        if matches:
            hybrid_answers.append(answer_structured(matches[0], field).answer)
            continue
    hybrid_answers.append("Informação não encontrada nos metadados estruturados.")

ids = [q["id"] for q in exact_questions]
expected = [q["resposta_esperada"] for q in exact_questions]

answer_rows = []
for name, answers in [("LLM sem RAG", no_rag_answers), ("TF-IDF puro", tfidf_answers),
                       ("BM25 puro", bm25_answers), ("Híbrido (estruturado)", hybrid_answers)]:
    s = evaluate_exact_answers(ids, answers, expected)
    answer_rows.append({"método": name, "taxa_correta": s.taxa_correta, "taxa_parcial": s.taxa_parcial,
                         "taxa_incorreta": s.taxa_incorreta, "taxa_nao_encontrada": s.taxa_nao_encontrada})

answer_summary = pd.DataFrame(answer_rows)
print("Acurácia em perguntas de campo exato (10 perguntas):")
answer_summary

Acurácia em perguntas de campo exato (10 perguntas):


,método,taxa_correta,taxa_parcial,taxa_incorreta,taxa_nao_encontrada
0,LLM sem RAG,0.2,0.0,0.8,0.0
1,TF-IDF puro,0.7,0.0,0.3,0.0
2,BM25 puro,0.9,0.0,0.1,0.0
3,Híbrido (estruturado),1.0,0.0,0.0,0.0


## 3. Resultados — Parte 2 (Autoencoder Gamma Ray)

Resumo dos resultados já obtidos e validados no notebook `03_gamma_ray_autoencoder.ipynb` (não reprocessado aqui por já envolver treino de rede neural, custoso; ver notebook 03 para reprodução completa).

| Métrica | Autoencoder Conv1D | PCA (8 comp.) | Média móvel |
|---|---|---|---|
| MSE médio (14 poços de teste, nunca vistos no treino) | **0.0004** | 0.0608 | 0.0474 |
| Poços em que venceu | **14/14** | 0/14 | 0/14 |

- **Dataset**: 82 poços com `.dlis`, 73 com curva Gamma Ray convencional utilizável (9 excluídos por só terem ferramentas especializadas sem canal GR de referência).
- **Split**: por poço (59 treino / 14 teste) — evita vazamento de informação entre janelas vizinhas do mesmo poço.
- **Validação de generalização**: o mesmo poço de teste, treinado isoladamente (split 80/20 interno, mesma arquitetura), obteve MSE ≈ 0.0044 — o modelo unificado (nunca viu esse poço) obteve MSE ≈ 0.0001 nele, confirmando que o ganho vem de mais dados de treino, não de vazamento.

## 4. Discussão crítica

### RAG documental

- **A hipótese central se confirmou**: o sistema híbrido (metadados estruturados + busca vetorial) supera claramente os três baselines em perguntas de campo exato (100% vs. 90% BM25, 70% TF-IDF, 20% LLM sem RAG). O padrão observado bate com o esperado: LLM sem contexto praticamente não acerta (alucina ou se recusa), BM25 supera TF-IDF (mais robusto a termos exatos como well_id), e o acesso direto aos metadados estruturados elimina qualquer ambiguidade de recuperação para esse tipo de pergunta.
- **Nas perguntas semânticas (sem well_id explícito no texto)**, os três métodos de recuperação (TF-IDF, BM25, vetorial) divergem significativamente entre si — evidência de que cada um captura sinais diferentes (léxico exato vs. similaridade semântica). Isso confirma a necessidade da arquitetura híbrida: nenhum método sozinho é suficiente para as duas classes de pergunta.
- **Limitação principal**: a qualidade da extração de metadados via OCR é o gargalo do sistema — campos como `bacia` e `formacao` só estão disponíveis em 66-70% dos poços, mesmo após três rodadas de correção de bugs de parsing (vazamento de rótulo de seção, valores fisicamente implausíveis). Isso limita a cobertura de perguntas de campo exato para poços com metadados incompletos, embora o sistema corretamente responda "não encontrado" nesses casos em vez de inventar.
- **Taxa de alucinação**: não medida automaticamente neste notebook (exigiria avaliação humana comparando cada resposta com a evidência recuperada) — mas o design do `qa_pipeline.py` (fallback por template + prompt rígido para o LLM) minimiza esse risco por construção, já que a resposta estruturada nunca é gerada livremente pelo modelo.

### Autoencoder Gamma Ray

- O Autoencoder Conv1D unificado superou PCA e média móvel em 14/14 poços de teste nunca vistos, com margem grande (MSE ~100x menor). O teste de sanidade (comparação com modelo treinado isoladamente no mesmo poço) descarta vazamento de dado como explicação — o ganho é genuíno e vem de treinar com mais poços agregados.
- Dois bugs reais de vazamento de dado foram identificados e corrigidos durante o desenvolvimento (interpolação sobre grandes lacunas ausentes; clipping de outliers agressivo demais) — documentados no notebook 03.
- O erro de reconstrução é um indicador de comportamento **estatisticamente incomum** da curva, não uma interpretação petrofísica — consistente com os limites definidos no escopo do projeto.

### Síntese geral

Os dois sistemas demonstram, de formas complementares, que técnicas de deep learning e representação (embeddings, busca híbrida, autoencoders convolucionais) trazem ganhos mensuráveis sobre baselines clássicos neste domínio — mas ambos são limitados pela qualidade do dado de entrada (OCR para o RAG; canal Gamma Ray disponível e "limpo" para o Autoencoder), reforçando que a engenharia de dados foi, na prática, tão importante quanto a escolha do modelo.